# Tinh chỉnh Trọng số Mô hình Thống Kê (ARIMA Family)

Notebook này tìm ra cấu hình tốt nhất cho các mô hình thống kê: **ARIMA, SARIMA, ARIMAX, SARIMAX**.
Các bước thực hiện:
1. **Lựa chọn Biến Ngoại sinh (Exogenous Feature Selection):** Sử dụng Spearman Correlation và Granger Causality.
2. **Kiểm tra Tính dừng (Stationarity):** Áp dụng Augmented Dickey-Fuller (ADF) Test.
3. **Auto-ARIMA (Hyperparameter Tuning):** Dùng thuật toán Grid Search (hoặc Stepwise) dựa trên tiêu chuẩn AIC.
4. **Kiểm định phần dư (Residual Diagnostics):** Sử dụng Ljung-Box Test để xác thực mô hình.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import pickle

from scipy.stats import spearmanr
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
import pmdarima as pm
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path("../outputs/predictions")
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODELING_DIR = Path("../data/processed/modeling_fs")
TARGET_LOG = "target_pm25_h24_log"


In [2]:
def load_xy(path: Path):
    df = pd.read_csv(path, parse_dates=["datetime_local"])
    feat = [c for c in df.columns if c not in ("datetime_local", TARGET_LOG)]
    return df[feat], df[TARGET_LOG], df["datetime_local"]

train_X, train_y, train_dt = load_xy(MODELING_DIR / "train_dl.csv")
val_X, val_y, val_dt = load_xy(MODELING_DIR / "val_dl.csv")
test_X, test_y, test_dt = load_xy(MODELING_DIR / "test_dl.csv")

print(f"Train shapes: X={train_X.shape}, y={train_y.shape}")
print(f"Val shapes: X={val_X.shape}, y={val_y.shape}")
print(f"Test shapes: X={test_X.shape}, y={test_y.shape}")


Train shapes: X=(6383, 26), y=(6383,)
Val shapes: X=(2128, 26), y=(2128,)
Test shapes: X=(2128, 26), y=(2128,)


## 1. Exogenous Feature Selection (Dành cho ARIMAX & SARIMAX)

### 1.1 Spearman Correlation
Tìm các biến (ngoại sinh) có tương quan đơn điệu tốt nhất với biến mục tiêu.

In [3]:
spearman_corr = {}
for col in train_X.columns:
    if "pm25" in col: # Loại bỏ các cột auto-correlation của PM2.5 vì mô hình ARIMA đã tự xử lý (AR)
        continue
    corr, pval = spearmanr(train_X[col], train_y)
    spearman_corr[col] = abs(corr)

corr_df = pd.DataFrame.from_dict(spearman_corr, orient='index', columns=['Spearman_Abs']).sort_values(by='Spearman_Abs', ascending=False)
display(corr_df.head(10))

# Lấy Top 5 biến weather tốt nhất để test vấp Granger
top_candidates = corr_df.head(5).index.tolist()
print("Top candidates:", top_candidates)


,Spearman_Abs
month_cos,0.468549
surface_pressure,0.393530
is_dry_season,0.360527
wind_u,0.312185
month,0.295046
temperature_2m,0.214018
relative_humidity_2m,0.109109
hour_cos,0.102689
hours_since_last_rain,0.053740
day_of_week,0.031981


Top candidates: ['month_cos', 'surface_pressure', 'is_dry_season', 'wind_u', 'month']


### 1.2 Granger Causality Test
Kiểm tra xem liệu các biến thời tiết có thực sự "gây ra" hay mang theo tín hiệu tiên lượng giá trị PM2.5 trong tương lai hay không. Ta sẽ kiểm chứng độ trễ từ 1 đến 24 giờ.

In [4]:
selected_exog = []
print("--- Kết quả Granger Causality (Tìm p-value < 0.05) ---")

for col in top_candidates:
    # Granger Causality yêu cầu DataFrame có 2 cột: [Cột mục tiêu (y), Cột nguyên nhân (X)]
    data = pd.concat([train_y, train_X[col]], axis=1)
    
    # maxlag=24 do ta quan tâm đến chu kỳ 1 ngày
    print(f"\nThử nghiệm: {col} -> PM2.5")
    try:
        gc_res = grangercausalitytests(data, maxlag=[1, 6, 12, 24], verbose=False)
        # Kiểm tra xem có bất kỳ độ trễ (lag) nào có p-value < 0.05 không
        has_causality = False
        for lag in gc_res:
            test_stat = gc_res[lag][0]['ssr_ftest']
            p_val = test_stat[1]
            if p_val < 0.05:
                has_causality = True
                break
        if has_causality:
            print(f" => {col} có quan hệ Granger Causality (p < 0.05).")
            selected_exog.append(col)
        else:
            print(f" => {col} KHÔNG có quan hệ Granger Causality.")
    except Exception as e:
         print(f" => Lỗi khi tính toán {col}: {e}")

print("\n=> Các biến ngoại sinh chính thức được chọn:", selected_exog)


--- Kết quả Granger Causality (Tìm p-value < 0.05) ---

Thử nghiệm: month_cos -> PM2.5
 => month_cos có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: surface_pressure -> PM2.5
 => surface_pressure có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: is_dry_season -> PM2.5
 => is_dry_season có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: wind_u -> PM2.5
 => wind_u có quan hệ Granger Causality (p < 0.05).

Thử nghiệm: month -> PM2.5
 => month có quan hệ Granger Causality (p < 0.05).

=> Các biến ngoại sinh chính thức được chọn: ['month_cos', 'surface_pressure', 'is_dry_season', 'wind_u', 'month']


In [5]:
# Trích xuất dữ liệu Exogenous
train_exog_vals = train_X[selected_exog].values
val_exog_vals = val_X[selected_exog].values
test_exog_vals = test_X[selected_exog].values

train_y_vals = train_y.values
val_y_vals = val_y.values
test_y_vals = test_y.values


## 2. Kiểm định tính dừng - Augmented Dickey-Fuller (ADF)

Chúng ta sử dụng thuật toán kiểm định chuỗi thời gian để quyết định bậc d (differencing).

In [6]:
adf_result = adfuller(train_y_vals)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("p-value < 0.05 => Bác bỏ giả thuyết H0 (Chuỗi Không Dừng). Chuỗi dữ liệu LÀ DỪNG (Stationary). Ta có thể xét d=0.")
else:
    print("p-value >= 0.05 => Không thể bác bỏ H0. Chuỗi dữ liệu LÀ KHÔNG DỪNG (Non-Stationary). Cần thiết lập d=1 hoặc d=2.")


ADF Statistic: -6.471124386274397
p-value: 1.3663498557108741e-08
p-value < 0.05 => Bác bỏ giả thuyết H0 (Chuỗi Không Dừng). Chuỗi dữ liệu LÀ DỪNG (Stationary). Ta có thể xét d=0.


## 3. Auto-ARIMA & Residual Diagnostics

Với dữ liệu `m=24`, tìm kiếm toàn bộ lưới (Grid Search) sẽ tốn rất nhiều giờ. Thay vào đó `stepwise=True` sử dụng thuật toán tham lam tìm đường gần nhất tới điểm tối ưu dựa trên AIC.

Chúng ta sẽ chạy 4 cấu hình:
- **ARIMA:** Khởi nghiệm nhanh, không có Chu kỳ (seasonal=False), không Ngoại sinh (exogenous=None).
- **SARIMA:** Có chu kỳ `m=24`, không Ngoại sinh.
- **ARIMAX:** Không chu kỳ, Có Ngoại sinh.
- **SARIMAX:** Có chu kỳ, Có Ngoại sinh.

> **Tips:** Tùy vào cấu hình máy, mỗi cell Search dưới đây có thể tốn từ 2 -> 15 phút.

In [7]:
# Hàm helper hỗ trợ đo lường Metric
def regression_metrics(y_true, y_pred, inverse=True):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if inverse:
        y_true_inv = np.expm1(y_true)
        y_pred_inv = np.expm1(y_pred)
    else:
        y_true_inv = y_true
        y_pred_inv = y_pred
    rmse = float(np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    eps = 1e-6
    mask = np.abs(y_true_inv) > eps
    mape = np.mean(np.abs((y_true_inv[mask] - y_pred_inv[mask]) / y_true_inv[mask])) * 100.0
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

# Từ vựng metric chung
all_metrics = []
predictions_dict = {} # Key: (Model_Name, split), Value: Array


In [8]:
# [1. ARIMA]
print("\n--- Tuning ARIMA (Non-Seasonal, No Exog) ---")
start = time.time()
model_arima = pm.auto_arima(train_y_vals, 
                            exogenous=None,
                            seasonal=False,
                            start_p=0, start_q=0, max_p=3, max_q=3,
                            d=None, trace=True, 
                            error_action='ignore', suppress_warnings=True, stepwise=True)
print(f"ARIMA tuned in {time.time()-start:.2f}s | Order: {model_arima.order} | AIC: {model_arima.aic():.2f}")

# Train and Predict (Dùng statsmodels API để linh hoạt áp dụng qua Val/Test)
sm_arima = sm.tsa.ARIMA(endog=train_y_vals, order=model_arima.order).fit()

predictions_dict[('ARIMA', 'train')] = sm_arima.predict(start=0, end=len(train_y_vals)-1)
predictions_dict[('ARIMA', 'val')] = sm_arima.apply(endog=val_y_vals).fittedvalues
predictions_dict[('ARIMA', 'test')] = sm_arima.apply(endog=test_y_vals).fittedvalues

# Diagnostics - Ljung-Box Test
lb_df = acorr_ljungbox(sm_arima.resid, lags=[24], return_df=True)
print("\nLjung-Box Test (ARIMA):")
display(lb_df)
print("=> Nếu p-value (lb_pvalue) > 0.05, phần dư chỉ là nhiễu trắng (mô hình đã bao quát tốt thông tin).")



--- Tuning ARIMA (Non-Seasonal, No Exog) ---
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-2469.853, Time=0.58 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=-2575.800, Time=0.51 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-2593.891, Time=0.58 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-2471.853, Time=0.23 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=-2601.977, Time=1.24 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=-3105.106, Time=4.11 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=-2628.522, Time=0.38 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=-3121.438, Time=6.22 sec
 ARIMA(3,1,0)(0,0,0)[0] intercept   : AIC=-2665.033, Time=0.65 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=-3115.365, Time=5.55 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=-2900.918, Time=5.18 sec
 ARIMA(3,1,1)(0,0,0)[0]             : AIC=-3123.454, Time=1.77 sec
 ARIMA(2,1,1)(0,0,0)[0]             : AIC=-3124.449, Time=1.16 sec
 ARIMA(1,1,1)(0,0,0)[0]             : AI

,lb_stat,lb_pvalue
24,194.325404,1.305172e-28


=> Nếu p-value (lb_pvalue) > 0.05, phần dư chỉ là nhiễu trắng (mô hình đã bao quát tốt thông tin).


In [9]:
# [2. ARIMAX]
print("\n--- Tuning ARIMAX (Non-Seasonal, With Exog) ---")
start = time.time()
model_arimax = pm.auto_arima(train_y_vals, 
                             X=train_exog_vals,
                             seasonal=False,
                             start_p=0, start_q=0, max_p=3, max_q=3,
                             d=None, trace=True, 
                             error_action='ignore', suppress_warnings=True, stepwise=True)
print(f"ARIMAX tuned in {time.time()-start:.2f}s | Order: {model_arimax.order} | AIC: {model_arimax.aic():.2f}")

sm_arimax = sm.tsa.ARIMA(endog=train_y_vals, exog=train_exog_vals, order=model_arimax.order).fit()

predictions_dict[('ARIMAX', 'train')] = sm_arimax.predict(start=0, end=len(train_y_vals)-1, exog=train_exog_vals)
predictions_dict[('ARIMAX', 'val')] = sm_arimax.apply(endog=val_y_vals, exog=val_exog_vals).fittedvalues
predictions_dict[('ARIMAX', 'test')] = sm_arimax.apply(endog=test_y_vals, exog=test_exog_vals).fittedvalues



--- Tuning ARIMAX (Non-Seasonal, With Exog) ---
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=-2576.389, Time=1.14 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=-2655.610, Time=2.15 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=-2672.226, Time=3.15 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=-2578.389, Time=1.28 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=-2685.550, Time=4.05 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=-3180.131, Time=7.14 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=-2721.008, Time=2.27 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=-3114.422, Time=8.55 sec
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=8.41 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=inf, Time=14.88 sec
 ARIMA(3,1,0)(0,0,0)[0] intercept   : AIC=-2759.646, Time=5.09 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=-3171.127, Time=8.95 sec
 ARIMA(2,1,1)(0,0,0)[0]             : AIC=-3183.430, Time=7.14 sec
 ARIMA(1,1,1)(0,0,0)[0]             : AIC=-2687.

c:\Users\User\Desktop\Matt Folder\Materials\MyUni\6th Semester\Business Data Analysis\Project\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [10]:
# [3. SARIMA]
print("\n--- Tuning SARIMA (Seasonal m=24, No Exog) ---")
# Giới hạn max_p, max_Q để tối ưu thời gian
start = time.time()
model_sarima = pm.auto_arima(train_y_vals, 
                             exogenous=None,
                             seasonal=True, m=24,
                             start_p=0, start_q=0, max_p=2, max_q=2,
                             start_P=0, start_Q=0, max_P=1, max_Q=1,
                             d=None, D=None, trace=True, 
                             error_action='ignore', suppress_warnings=True, stepwise=True)
print(f"SARIMA tuned in {time.time()-start:.2f}s | Order: {model_sarima.order} | Seasonal: {model_sarima.seasonal_order} | AIC: {model_sarima.aic():.2f}")

sm_sarima = sm.tsa.statespace.SARIMAX(endog=train_y_vals, order=model_sarima.order, seasonal_order=model_sarima.seasonal_order).fit(disp=False)

predictions_dict[('SARIMA', 'train')] = sm_sarima.fittedvalues
predictions_dict[('SARIMA', 'val')] = sm_sarima.apply(endog=val_y_vals).fittedvalues
predictions_dict[('SARIMA', 'test')] = sm_sarima.apply(endog=test_y_vals).fittedvalues



--- Tuning SARIMA (Seasonal m=24, No Exog) ---
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[24] intercept   : AIC=-2469.853, Time=0.54 sec
 ARIMA(1,1,0)(1,0,0)[24] intercept   : AIC=-2660.595, Time=3.72 sec
 ARIMA(0,1,1)(0,0,1)[24] intercept   : AIC=-2663.327, Time=8.57 sec
 ARIMA(0,1,0)(0,0,0)[24]             : AIC=-2471.853, Time=0.42 sec
 ARIMA(0,1,1)(0,0,0)[24] intercept   : AIC=-2593.891, Time=0.99 sec
 ARIMA(0,1,1)(1,0,1)[24] intercept   : AIC=inf, Time=30.39 sec
 ARIMA(0,1,1)(1,0,0)[24] intercept   : AIC=-2675.566, Time=12.15 sec
 ARIMA(0,1,0)(1,0,0)[24] intercept   : AIC=-2588.447, Time=1.90 sec
 ARIMA(1,1,1)(1,0,0)[24] intercept   : AIC=-2687.668, Time=10.00 sec
 ARIMA(1,1,1)(0,0,0)[24] intercept   : AIC=-2601.977, Time=1.21 sec
 ARIMA(1,1,1)(1,0,1)[24] intercept   : AIC=inf, Time=27.99 sec
 ARIMA(1,1,1)(0,0,1)[24] intercept   : AIC=-2674.518, Time=11.19 sec
 ARIMA(2,1,1)(1,0,0)[24] intercept   : AIC=-3202.107, Time=44.62 sec
 ARIMA(2,1,1)(0,0,0)[24] interc

In [11]:
# [4. SARIMAX]
print("\n--- Tuning SARIMAX (Seasonal m=24, With Exog) ---")
start = time.time()
model_sarimax = pm.auto_arima(train_y_vals, 
                              X=train_exog_vals,
                              seasonal=True, m=24,
                              start_p=0, start_q=0, max_p=2, max_q=2,
                              start_P=0, start_Q=0, max_P=1, max_Q=1,
                              d=None, D=None, trace=True, 
                              error_action='ignore', suppress_warnings=True, stepwise=True)
print(f"SARIMAX tuned in {time.time()-start:.2f}s | Order: {model_sarimax.order} | Seasonal: {model_sarimax.seasonal_order} | AIC: {model_sarimax.aic():.2f}")

sm_sarimax = sm.tsa.statespace.SARIMAX(endog=train_y_vals, exog=train_exog_vals, order=model_sarimax.order, seasonal_order=model_sarimax.seasonal_order).fit(disp=False)

predictions_dict[('SARIMAX', 'train')] = sm_sarimax.fittedvalues
predictions_dict[('SARIMAX', 'val')] = sm_sarimax.apply(endog=val_y_vals, exog=val_exog_vals).fittedvalues
predictions_dict[('SARIMAX', 'test')] = sm_sarimax.apply(endog=test_y_vals, exog=test_exog_vals).fittedvalues

lb_df_sarimax = acorr_ljungbox(sm_sarimax.resid, lags=[24], return_df=True)
print("\nLjung-Box Test (SARIMAX):")
display(lb_df_sarimax)



--- Tuning SARIMAX (Seasonal m=24, With Exog) ---
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[24] intercept   : AIC=-2576.389, Time=2.06 sec
 ARIMA(1,1,0)(1,0,0)[24] intercept   : AIC=-2719.686, Time=28.85 sec
 ARIMA(0,1,1)(0,0,1)[24] intercept   : AIC=-2725.589, Time=32.61 sec
 ARIMA(0,1,0)(0,0,0)[24]             : AIC=-2578.389, Time=2.10 sec
 ARIMA(0,1,1)(0,0,0)[24] intercept   : AIC=-2672.226, Time=4.86 sec
 ARIMA(0,1,1)(1,0,1)[24] intercept   : AIC=inf, Time=65.10 sec
 ARIMA(0,1,1)(1,0,0)[24] intercept   : AIC=-2733.466, Time=47.14 sec
 ARIMA(0,1,0)(1,0,0)[24] intercept   : AIC=-2661.411, Time=33.64 sec
 ARIMA(1,1,1)(1,0,0)[24] intercept   : AIC=-2749.203, Time=43.80 sec
 ARIMA(1,1,1)(0,0,0)[24] intercept   : AIC=-2685.550, Time=6.91 sec
 ARIMA(1,1,1)(1,0,1)[24] intercept   : AIC=inf, Time=75.95 sec
 ARIMA(1,1,1)(0,0,1)[24] intercept   : AIC=-2740.857, Time=54.88 sec
 ARIMA(2,1,1)(1,0,0)[24] intercept   : AIC=-3184.808, Time=76.97 sec
 ARIMA(2,1,1)(0,0,0)[24] 

c:\Users\User\Desktop\Matt Folder\Materials\MyUni\6th Semester\Business Data Analysis\Project\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Ljung-Box Test (SARIMAX):


,lb_stat,lb_pvalue
24,83.019187,1.995694e-08


## 4. Leaderboard Nhóm Thống Kê & Lưu Xuất

So sánh thông số thực tế của cấu hình tối ưu thu được và lưu file dưới dạng numpy Array `.npy`

In [12]:
for name in ['ARIMA', 'ARIMAX', 'SARIMA', 'SARIMAX']:
    all_metrics.append({'model': name, 'split': 'train', **regression_metrics(train_y_vals, predictions_dict[(name, 'train')])})
    all_metrics.append({'model': name, 'split': 'val', **regression_metrics(val_y_vals, predictions_dict[(name, 'val')])})
    all_metrics.append({'model': name, 'split': 'test', **regression_metrics(test_y_vals, predictions_dict[(name, 'test')])})

metric_df = pd.DataFrame(all_metrics)

print("--- LEADERBOARD BẢN TUNED TRÊN TEST-SET ---")
display(metric_df[metric_df['split'] == 'test'].set_index('model')[['RMSE', 'MAE', 'MAPE']].round(4))

# Cài đặt mô hình siêu việt nhất thuộc nhóm Stats
best_stat_model = "SARIMAX" # Thông thường thuộc SARIMAX/SARIMA.


--- LEADERBOARD BẢN TUNED TRÊN TEST-SET ---


,RMSE,MAE,MAPE
model,,,
ARIMA,9.5774,5.8598,14.1587
ARIMAX,9.6369,5.9325,14.2782
SARIMA,9.4573,5.7647,13.9244
SARIMAX,9.3119,5.6027,13.4900


In [14]:
# Export To Pickle (Dictionaries with arrays)
preds_export = {
    "arima": {
        "train": predictions_dict[("ARIMA", "train")],
        "val": predictions_dict[("ARIMA", "val")],
        "test": predictions_dict[("ARIMA", "test")],
    },
    "arimax": {
        "train": predictions_dict[("ARIMAX", "train")],
        "val": predictions_dict[("ARIMAX", "val")],
        "test": predictions_dict[("ARIMAX", "test")],
    },
    "sarima": {
        "train": predictions_dict[("SARIMA", "train")],
        "val": predictions_dict[("SARIMA", "val")],
        "test": predictions_dict[("SARIMA", "test")],
    },
    "sarimax": {
        "train": predictions_dict[("SARIMAX", "train")],
        "val": predictions_dict[("SARIMAX", "val")],
        "test": predictions_dict[("SARIMAX", "test")],
    }
}

pickle_path = PRED_DIR / "tuned_stats_preds.pkl"
with open(pickle_path, 'wb') as handle:
    pickle.dump(preds_export, handle, protocol=pickle.HIGHEST_PROTOCOL)
